In [1]:
import chromadb

CHROMA_PATH = "./chroma_db"
COLLECTION_NAME = "tesis_docs"

client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_collection(COLLECTION_NAME)

# 1. Cuántos chunks hay en total
print(f"Total de chunks en la colección: {collection.count()}")
print("=" * 50)

# 2. Ver una muestra (primeros N) con su texto y metadata
sample = collection.peek(limit=5)
for i in range(len(sample["ids"])):
    print(f"ID: {sample['ids'][i]}")
    print(f"Metadata: {sample['metadatas'][i]}")
    print(f"Texto: {sample['documents'][i][:200]}...")
    print("-" * 50)

# 3. Ver cuántos chunks hay por archivo (verificar que todos los PDFs entraron)
all_data = collection.get(include=["metadatas"])
from collections import Counter
counts = Counter(m["file_name"] for m in all_data["metadatas"])
print("Chunks por archivo:")
for file_name, n in counts.items():
    print(f"  {file_name}: {n} chunks")

Total de chunks en la colección: 2158
ID: 01_Practicas-y-politicas-para-reducir-los-sesgos-sobre-el-peso-en-el-manejo-de-la-obesidad-1.pdf_chunk_0
Metadata: {'file_name': '01_Practicas-y-politicas-para-reducir-los-sesgos-sobre-el-peso-en-el-manejo-de-la-obesidad-1.pdf', 'chunk_index': 0}
Texto: <!-- image -->

## Prácticas y políticas para reducir los sesgos sobre el peso en el manejo de la obesidad

2022 adaptado por: Salvo Cofman K i ,ii , Leiva Velasco M iii , Gómez -Pérez D iv , Oda-Mont...
--------------------------------------------------
ID: 01_Practicas-y-politicas-para-reducir-los-sesgos-sobre-el-peso-en-el-manejo-de-la-obesidad-1.pdf_chunk_1
Metadata: {'chunk_index': 1, 'file_name': '01_Practicas-y-politicas-para-reducir-los-sesgos-sobre-el-peso-en-el-manejo-de-la-obesidad-1.pdf'}
Texto: - i) Departamento de Medicina Interna, Centro de Nutrición y Diabetes. Clínica Alemana de Santiago, Chile.
2. ii) Facultad de Medicina, Universidad del Desarrollo, Santiago, Chile.
3. iii) De

In [2]:
from openai import OpenAI

client_lm = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

def embed_query(text, model="text-embedding-qwen3-embedding-8b"):
    instruct_text = (
        "Instruct: Given a web search query, retrieve relevant passages that answer the query.\n"
        f"Query: {text}"
    )
    response = client_lm.embeddings.create(input=[instruct_text], model=model)
    return response.data[0].embedding

query = "¿de qué trata el documento?"  # cambia por una pregunta real de tu dominio
query_embedding = embed_query(query)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
)

for i in range(len(results["ids"][0])):
    print(f"Score (distancia): {results['distances'][0][i]:.4f}")
    print(f"Archivo: {results['metadatas'][0][i]['file_name']}")
    print(f"Texto: {results['documents'][0][i][:300]}...")
    print("-" * 50)

Score (distancia): 1.2053
Archivo: 10_Intervenciones-psicologicas-y-conductuales-eficaces-en-el-tratamiento-de-la-obesidad.pdf
Texto: ## Cómo citar este documento

Intervenciones psicológicas y conductuales eficaces en el tratamiento de la obesidad. Adaptación de la guía de práctica clínica (Coalición chilena para el estudio de la obesidad, version 1, 2022) por Arias E, Assad V, Vargas G. Capítulo adaptado de: Vallis TM, Macklin D...
--------------------------------------------------
Score (distancia): 1.2124
Archivo: 16_Cirugia-bariatrica-Tratamiento-postoperatorio.pdf
Texto: .                                                                                                                                                                                                                                                                                                           ...
--------------------------------------------------
Score (distancia): 1.2157
Archivo: 08_Terapia-de-nutricion-me